# Demo: Place One Gemini Prediction Market Order

Minimal API demo using `GeminiTrader`.

- Finds one currently active contract
- Chooses a side/ask
- Places one IOC limit buy order


In [ ]:
import os
import requests
from datetime import datetime, timezone

from btc_hourly_model import parse_event_ticker
from gemini_trader import GeminiTrader


In [ ]:
# --- Config ---
API_KEY = os.getenv("GEMINI_API_KEY", "")
API_SECRET = os.getenv("GEMINI_API_SECRET", "")
SANDBOX = True  # True for paper test, False for real money
BET_DOLLARS = 5.0
PLACE_ORDER = False  # Set True to actually send the order

if not API_KEY or not API_SECRET:
    raise ValueError("Set GEMINI_API_KEY and GEMINI_API_SECRET first.")

trader = GeminiTrader(
    api_key=API_KEY,
    api_secret=API_SECRET,
    sandbox=SANDBOX,
    bet_dollars=BET_DOLLARS,
)

print(f"Trader ready. sandbox={SANDBOX}  bet=${BET_DOLLARS}")


In [ ]:
# Discover one live contract with an actionable ask
base_url = "https://api.sandbox.gemini.com" if SANDBOX else "https://api.gemini.com"
resp = requests.get(f"{base_url}/v1/prediction-markets/events", timeout=15)
resp.raise_for_status()
data = resp.json()
events = data.get("data", data) if isinstance(data, dict) else data

now_utc = datetime.now(timezone.utc)
candidate = None

for event in events:
    ticker = event.get("ticker", "")
    asset, settle_utc = parse_event_ticker(ticker)
    if asset is None:
        continue
    if (settle_utc - now_utc).total_seconds() <= 0:
        continue

    for c in event.get("contracts", []):
        prices = c.get("prices", {})
        ask_yes = prices.get("buy", {}).get("yes")
        ask_no = prices.get("buy", {}).get("no")

        # pick whichever side has a valid positive ask
        side, ask = None, None
        if ask_yes is not None and float(ask_yes) > 0:
            side, ask = "YES", float(ask_yes)
        elif ask_no is not None and float(ask_no) > 0:
            side, ask = "NO", float(ask_no)

        if side is not None:
            candidate = {
                "event_ticker": ticker,
                "contract_id": str(c.get("id", "")),
                "label": c.get("label", ""),
                "side": side,
                "ask": ask,
            }
            break

    if candidate is not None:
        break

if candidate is None:
    raise RuntimeError("No tradable prediction contract found right now.")

candidate


In [ ]:
# Place one order (IOC limit buy) with optional aggressive pricing

AGGRESSIVE_PREMIUM = 0.0   # default: trade at live ask (no extra premium)
MAX_RETRIES = 3

def fetch_current_ask(base_url, event_ticker, contract_id, side):
    r = requests.get(f"{base_url}/v1/prediction-markets/events", timeout=15)
    r.raise_for_status()
    data = r.json()
    events = data.get("data", data) if isinstance(data, dict) else data
    for event in events:
        if event.get("ticker") != event_ticker:
            continue
        for c in event.get("contracts", []):
            if str(c.get("id", "")) != str(contract_id):
                continue
            prices = c.get("prices", {})
            key = "yes" if side == "YES" else "no"
            raw_ask = prices.get("buy", {}).get(key)
            return float(raw_ask) if raw_ask is not None else None
    return None

if not PLACE_ORDER:
    print("PLACE_ORDER=False -> no order sent.")
    print("Set PLACE_ORDER=True and re-run this cell to submit.")
else:
    result = None
    for attempt in range(1, MAX_RETRIES + 1):
        live_ask = fetch_current_ask(base_url, candidate["event_ticker"], candidate["contract_id"], candidate["side"])
        if live_ask is None:
            print(f"Attempt {attempt}: live ask unavailable, retrying...")
            continue

        limit_price = min(1.0, round(live_ask + AGGRESSIVE_PREMIUM, 4))
        print(f"Attempt {attempt}: side={candidate[side]} live_ask={live_ask:.4f} limit={limit_price:.4f}")

        result = trader.place_order(
            contract_id=candidate["contract_id"],
            side=candidate["side"],
            ask_price=live_ask,
            limit_price=limit_price,
        )
        print(result)

        if result.get("filled", 0) > 0:
            print("Filled successfully.")
            break
        print("No fill on this IOC attempt.")

    if result is None or result.get("filled", 0) == 0:
        print("Order not filled. Try smaller size, higher premium, or a more liquid contract.")


In [ ]:
# Check order result via positions snapshot (status endpoint is unavailable)
if "result" not in globals() or not result.get("symbol"):
    print("No order result found. Submit an order first with PLACE_ORDER=True.")
else:
    snap = trader.get_order_status(result.get("order_id", -1))
    all_positions = snap.get("positions", [])
    symbol = result["symbol"]
    matched = [p for p in all_positions if str(p.get("symbol", "")) == str(symbol)]
    print({"symbol": symbol, "matched_positions": matched, "raw_snapshot": snap})


In [ ]:
import pandas as pd

# ── Account balance (cash available) ─────────────────────────────────────────
balances = trader.get_balances()
bal_rows = [
    {"currency": b["currency"], "total": b["amount"], "available": b["available"]}
    for b in balances
    if float(b.get("amount", 0)) > 0
]
print("=== ACCOUNT BALANCES ===")
display(pd.DataFrame(bal_rows).set_index("currency"))

# ── Open prediction-market positions ─────────────────────────────────────────
positions = trader.get_positions()
print(f"\n=== OPEN CONTRACTS  ({len(positions)} positions) ===")
if positions:
    pos_rows = []
    for p in positions:
        meta = p.get("contractMetadata", {})
        qty      = float(p.get("totalQuantity", 0))
        avg_cost = float(p.get("avgPrice", 0))
        pos_rows.append({
            "symbol":     p.get("instrumentSymbol") or meta.get("instrumentSymbol", "?"),
            "outcome":    p.get("outcome", "?").upper(),
            "qty":        qty,
            "avg_cost":   avg_cost,
            "cost_basis": round(qty * avg_cost, 4),   # total $ spent on this position
        })
    display(pd.DataFrame(pos_rows))
else:
    print("  No open positions.")